# Just Think: replication on Qwen 2.5 32B Instruct

This notebook runs the preregistered replication (arms A, C and D, 100 trials each) on a Kaggle GPU. It needs the **GPU T4 x2** accelerator and **Internet on**. A session lasts at most 12 hours, so the run stops itself after 11 hours and can be resumed in a new session. Read `kaggle/README.md` in the repository first.

In [ ]:
# Settings. REPO_URL is the public GitHub repository of this project.
REPO_URL = "https://github.com/<your-username>/just-think"
MODEL = "qwen32b-nf4"
PLAN = "replication"
MAX_HOURS = 11.0
BATCH = 4

In [ ]:
import os, subprocess
if not os.path.exists("just-think"):
    subprocess.run(["git", "clone", REPO_URL, "just-think"], check=True)
os.chdir("just-think")
subprocess.run(["git", "clone", "https://github.com/valen-research/Pain-axis", "external/Pain-axis"], check=False)
subprocess.run(["git", "-C", "external/Pain-axis", "checkout", "4d75cd90e206ea962f7a9101e65c85efea56723b"], check=True)
print(subprocess.run(["git", "log", "-1", "--format=%H"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q "transformers>=4.46" "accelerate>=1.0" bitsandbytes statsmodels sentence-transformers

In [ ]:
# Resume: if an earlier session's output was added as an input dataset, copy its results in.
import glob, shutil
for src in glob.glob(f"/kaggle/input/*/just-think-results/{MODEL}"):
    shutil.copytree(src, f"results/{MODEL}", dirs_exist_ok=True)
    print("resumed from", src)
!ls -R results 2>/dev/null | head -20

In [ ]:
import torch
print(torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## Calibration

This runs once. It checks the vector, builds the pleasant vector and picks the dose with the preregistered rule. It is skipped when a calibration already exists.

In [ ]:
if not os.path.exists(f"results/{MODEL}/calibration/calibration.json"):
    !python scripts/calibrate.py {MODEL}
!cat results/{MODEL}/calibration/calibration.json

## Trials

Trials are written as they finish, so nothing is lost if the session ends.

In [ ]:
!python scripts/run_study.py {MODEL} --plan {PLAN} --batch {BATCH} --max-hours {MAX_HOURS}

## Save

The results are copied to `/kaggle/working/just-think-results`, which Kaggle keeps as the notebook output. To resume, add this notebook's output as an input dataset in the next session and run all cells again.

In [ ]:
shutil.copytree("results", "/kaggle/working/just-think-results", dirs_exist_ok=True)
!wc -l /kaggle/working/just-think-results/{MODEL}/trials/*.jsonl